ChatGPT API를 이용한 txt 파일 요약 과정

In [7]:
! pip install --upgrade google-generativeai


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
! pip show google-generativeai

Name: google-generativeai
Version: 0.8.6
Summary: Google Generative AI High level API client library and tools.
Home-page: https://github.com/google/generative-ai-python
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: google-ai-generativelanguage, google-api-core, google-api-python-client, google-auth, protobuf, pydantic, tqdm, typing-extensions
Required-by: 


In [13]:
import google.generativeai as genai
import os
import glob
import pandas as pd
from tqdm import tqdm

# API 키 설정 (환경 변수 사용)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

In [ ]:
def load_texts_from_folder(folder_path):
    texts = []

    for file in glob.glob(os.path.join(folder_path, "*.txt")):
        with open(file, encoding="utf-8") as f:
            content = f.read().strip()
            if content:
                texts.append(f"[파일: {os.path.basename(file)}]\n{content}")

    return "\n\n".join(texts)

DefaultCredentialsError: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.

In [ ]:
def split_text(text, max_len=1200):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    chunks, current = [], ""

    for line in lines:
        if len(current) + len(line) <= max_len:
            current += " " + line
        else:
            chunks.append(current.strip())
            current = line

    if current:
        chunks.append(current.strip())

    return chunks

In [ ]:
base_dir = "변환됨"
folders = [
    "국기연_발간물",
    "국방위_국정감사",
    "국방위_보도",
    "국방위_회의록"
]

for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    full_text = load_texts_from_folder(folder_path)
    chunks = split_text(full_text)

    results = []

    for i, chunk in enumerate(tqdm(chunks, desc=folder)):
        summary = gemini_clean_summary(chunk)

        if "의미 있는 정보 없음" in summary:
            continue

        results.append({
            "chunk_id": i,
            "clean_summary": summary
        })

    df = pd.DataFrame(results)
    df.to_csv(f"{folder}_summary.csv", index=False, encoding="utf-8-sig")

    print(f"✅ {folder}_summary.csv 생성 완료")